In [0]:
from pyspark.sql.functions import col, when, current_timestamp, lit, udf
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import mlflow
import pandas as pd

# Load gold feature table
df_gold = spark.table("gold_db.gold_bank_features").fillna(0)

print("Gold table loaded:", df_gold.count(), "banks")
display(df_gold.limit(3))

Gold table loaded: 108 banks


CERT,failed,avg_asset,avg_dep,avg_equity,avg_net_income,avg_capital_ratio,avg_loan_to_deposit,avg_profit_margin,avg_leverage_ratio,avg_deposit_to_asset,pagerank,degree_centrality,betweenness_centrality,clustering_coefficient,asset_volatility,income_volatility,risk_tier
10004,0,81541.51785714286,67218.08333333333,9569.452380952382,412.32142857142856,0.11874583333333338,0.7307970238095235,0.00538095238095238,8.72588452380952,0.832798809523809,0.008770303993570245,0.07476635514018659,0.12454631623531502,0.642857142857142,50567.17512307588,325.1971179547732,LOW
10013,0,157259.5294117647,131139.0,13445.823529411764,1204.2156862745098,0.08050196078431372,0.7765705882352941,0.0066745098039215685,12.612827450980394,0.8585333333333335,0.007845867545949734,0.07476635514018691,0.09725910709938193,0.6428571428571427,84874.06056654833,1115.2227636436678,MEDIUM
10119,0,17253.880597014926,15075.477611940298,1810.910447761194,46.59701492537314,0.09310149253731342,0.6523999999999998,0.0031761194029850752,12.027438805970151,0.8857820895522388,0.014436710277083242,0.07476635514018691,0.021608154747142432,0.6428571428571435,11961.189920691848,104.7154961907237,MEDIUM


In [0]:
# Define feature columns
feature_cols = [
    "avg_asset", "avg_dep", "avg_equity", "avg_net_income",
    "avg_capital_ratio", "avg_loan_to_deposit", "avg_profit_margin",
    "avg_leverage_ratio", "avg_deposit_to_asset",
    "pagerank", "degree_centrality", "betweenness_centrality",
    "clustering_coefficient", "asset_volatility", "income_volatility"
]

# Assemble features
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_ml = assembler.transform(df_gold).select("CERT", "features", "failed", "risk_tier")

# Handle class imbalance
df_active = df_ml.filter("failed = 0")
df_failed = df_ml.filter("failed = 1")
df_failed_oversampled = df_failed.sample(withReplacement=True, fraction=5.0, seed=42)
df_balanced = df_active.union(df_failed_oversampled)

# Train model on full balanced dataset
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="failed",
    numTrees=100,
    maxDepth=5,
    seed=42
)
model = rf.fit(df_balanced)

print("Model retrained on full dataset!")
print("Total banks used for training:", df_balanced.count())

Model retrained on full dataset!
Total banks used for training: 129


In [0]:
# Run predictions on all 108 banks
df_all_features = assembler.transform(df_gold).select(
    "CERT", "features", "failed", "risk_tier",
    "avg_asset", "avg_capital_ratio", "avg_leverage_ratio",
    "pagerank", "betweenness_centrality"
)

# Get predictions
batch_predictions = model.transform(df_all_features)

# Extract failure probability
get_failure_prob = udf(lambda v: float(v[1]), DoubleType())

df_inference = batch_predictions \
    .withColumn("failure_probability", get_failure_prob("probability")) \
    .withColumn("contagion_risk", 
        when(col("failure_probability") >= 0.8, "CRITICAL")
        .when(col("failure_probability") >= 0.6, "HIGH")
        .when(col("failure_probability") >= 0.4, "MEDIUM")
        .otherwise("LOW")
    ) \
    .withColumn("inference_timestamp", current_timestamp()) \
    .withColumn("model_version", lit("random_forest_v1")) \
    .select(
        "CERT", "risk_tier", "failed",
        "failure_probability", "contagion_risk",
        "avg_asset", "avg_capital_ratio",
        "avg_leverage_ratio", "pagerank",
        "betweenness_centrality",
        "inference_timestamp", "model_version"
    )

print("Batch inference complete!")
print("Total banks scored:", df_inference.count())
print("\nContagion risk distribution:")
df_inference.groupBy("contagion_risk").count().orderBy("count", ascending=False).show()

Batch inference complete!
Total banks scored: 108

Contagion risk distribution:
+--------------+-----+
|contagion_risk|count|
+--------------+-----+
|           LOW|  100|
|      CRITICAL|    7|
|          HIGH|    1|
+--------------+-----+



In [0]:
df_inference.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("gold_db.gold_inference_results")

print("gold_inference_results saved:", spark.table("gold_db.gold_inference_results").count(), "rows")

# Show top 10 most dangerous banks
print("\nTop 10 Most Dangerous Banks:")
display(
    spark.table("gold_db.gold_inference_results")
    .orderBy("failure_probability", ascending=False)
    .limit(10)
)

gold_inference_results saved: 108 rows

Top 10 Most Dangerous Banks:


CERT,risk_tier,failed,failure_probability,contagion_risk,avg_asset,avg_capital_ratio,avg_leverage_ratio,pagerank,betweenness_centrality,inference_timestamp,model_version
10085,CRITICAL,1,0.9977714285714286,CRITICAL,14460.333333333334,0.04805,11.097850000000001,0.011563152088114923,0.018703749870788464,2026-03-13T17:20:05.259Z,random_forest_v1
10086,HIGH,1,0.978586975343497,CRITICAL,133159.73394495412,0.07500366972477059,14.185927522935774,0.008661905076631891,0.10545859110185583,2026-03-13T17:20:05.259Z,random_forest_v1
10116,MEDIUM,1,0.9770608891108892,CRITICAL,14418.857142857143,0.08270000000000001,12.363457142857142,0.00918296727688933,0.018146077224631758,2026-03-13T17:20:05.259Z,random_forest_v1
10054,MEDIUM,1,0.9763073417987472,CRITICAL,170200.3173076923,0.1001846153846154,10.63728269230769,0.007302859347841807,0.08698216576775956,2026-03-13T17:20:05.259Z,random_forest_v1
10094,HIGH,1,0.9477714285714286,CRITICAL,37756.46153846154,0.057061538461538455,-8.133038461538462,0.011041446628460665,0.08698216576775972,2026-03-13T17:20:05.259Z,random_forest_v1
10155,MEDIUM,1,0.9212106497523666,CRITICAL,31369.3125,0.08999375,7.6420812499999995,0.009372319221798845,0.07608235834723899,2026-03-13T17:20:05.259Z,random_forest_v1
1006,MEDIUM,1,0.8758502603461656,CRITICAL,102857.92233009709,0.09231941747572818,11.923082524271841,0.007124272804838961,0.11455665452116778,2026-03-13T17:20:05.259Z,random_forest_v1
10100,MEDIUM,1,0.7280670995670997,HIGH,1976400.3039215687,0.0836980392156862,12.649467647058822,0.01375493102796586,8.751638489141471E-5,2026-03-13T17:20:05.259Z,random_forest_v1
10153,HIGH,0,0.1674306478405316,LOW,959962.6666666666,0.06199629629629629,16.222255555555552,0.0140389497323517,0.0024917581068499344,2026-03-13T17:20:05.259Z,random_forest_v1
10093,CRITICAL,0,0.1397266004683172,LOW,162915.76,0.06967733333333335,20.89233466666666,0.007451158661953488,0.08774071349090035,2026-03-13T17:20:05.259Z,random_forest_v1
